# Анализ Gating Weights (α) — DGCA Fusion на DUSHA

BERT загружается из `.safetensors` директории, WavLM и Fusion из `.pt` файлов.  
Текст берётся из колонки `speaker_text` TSV (Whisper не нужен).

In [ ]:
import subprocess, sys, os
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'peft>=0.10', 'soundfile', 'librosa',
    'scikit-learn', 'tqdm', 'pyyaml', 'sentencepiece',
], check=True)

REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Done.')

In [ ]:
import warnings, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import soundfile as sf
import librosa
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoFeatureExtractor
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── пути (Kaggle input) ───────────────────────────────────────────────────────
BERT_CKPT_DIR = '/kaggle/input/datasets/aleksandribryanov/wavlm-dusha-checkpoints/best_bert_dusha'
WAVLM_CKPT    = '/kaggle/input/datasets/aleksandribryanov/wavlm-dusha-checkpoints/wavlm_dusha_majority.pt'
FUSION_PT     = '/kaggle/input/datasets/aleksandribryanov/wavlm-dusha-checkpoints/best_dgca_fusion.pt'

AGG_ROOT   = Path('/kaggle/input/datasets/aleksandribryanov/agg-dusha')
TEST_TSV   = AGG_ROOT / 'aggregated_ds_0.9_test.tsv'
AUDIO_TEST = Path('/kaggle/input/datasets/sigireddybalasai/dusha-datasetcrowd/crowd_test')
OUT_DIR    = Path('/kaggle/working')

SR_TARGET    = 16_000
MAX_TEXT_LEN = 128
MAX_AUDIO_S  = 10.0
NUM_HEADS    = 4
BATCH_SIZE   = 16

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

DUSHA_LABEL2ID = {'neutral': 0, 'angry': 1, 'positive': 2, 'sad': 3, 'other': 4}
DUSHA_LABELS   = ['neutral', 'angry', 'positive', 'sad', 'other']
NUM_CLASSES    = 5

## Загрузка моделей

In [ ]:
from src.models import build_model
from src.config import ExperimentConfig

# BERT из .safetensors директории
for wrong, right in [('config (1).json', 'config.json')]:
    p = Path(BERT_CKPT_DIR) / wrong
    if p.exists() and not (Path(BERT_CKPT_DIR) / right).exists():
        shutil.copy(p, Path(BERT_CKPT_DIR) / right)

print('Loading BERT from', BERT_CKPT_DIR)
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_CKPT_DIR)
bert_backbone  = AutoModel.from_pretrained(BERT_CKPT_DIR).to(device)
bert_backbone.eval()
BERT_DIM = bert_backbone.config.hidden_size
print(f'  hidden_size: {BERT_DIM}')

# WavLM из .pt
print('Loading WavLM from', WAVLM_CKPT)
ckpt_wavlm = torch.load(WAVLM_CKPT, map_location=device, weights_only=False)
wavlm_cfg  = ExperimentConfig()
for k, v in ckpt_wavlm.get('config', {}).items():
    if hasattr(wavlm_cfg, k): setattr(wavlm_cfg, k, v)
wavlm_full = build_model(wavlm_cfg)
wavlm_full.load_state_dict(ckpt_wavlm['model_state_dict'])
wavlm_full.to(device)
wavlm_backbone  = wavlm_full.backbone
wavlm_processor = AutoFeatureExtractor.from_pretrained(wavlm_cfg.processor_name or wavlm_cfg.model_name)
wavlm_backbone.eval()
WAVLM_DIM = wavlm_backbone.config.hidden_size
print(f'  hidden_size: {WAVLM_DIM}')

In [ ]:
class DGCAFusion(nn.Module):
    def __init__(self, d_text, d_audio, D, num_heads, num_classes, dropout=0.0):
        super().__init__()
        self.proj_text  = nn.Linear(d_text,  D)
        self.proj_audio = nn.Linear(d_audio, D)
        self.mha_t2a    = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.mha_a2t    = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.ln_text    = nn.LayerNorm(D)
        self.ln_audio   = nn.LayerNorm(D)
        self.gate_text  = nn.Linear(D, D)
        self.gate_audio = nn.Linear(D, D)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(D, num_classes)

    def forward_with_alpha(self, h_text, h_audio):
        T = self.proj_text(h_text).unsqueeze(1)
        A = self.proj_audio(h_audio).unsqueeze(1)
        T_ref = self.ln_text(T  + self.mha_t2a(T, A, A)[0]).squeeze(1)
        A_ref = self.ln_audio(A + self.mha_a2t(A, T, T)[0]).squeeze(1)
        gates   = F.softmax(torch.stack([self.gate_text(T_ref), self.gate_audio(A_ref)], dim=-1), dim=-1)
        alpha_t = gates[..., 0]
        fused   = alpha_t * T_ref + (1 - alpha_t) * A_ref
        return self.classifier(self.dropout(fused)), alpha_t


print('Loading fusion from', FUSION_PT)
ckpt_f     = torch.load(FUSION_PT, map_location=device, weights_only=False)
FUSION_DIM = ckpt_f['fusion']['proj_text.weight'].shape[0]
print(f'  step={ckpt_f["step"]}  val_wacc={ckpt_f["val_wacc"]:.4f}  D={FUSION_DIM}')

fusion = DGCAFusion(BERT_DIM, WAVLM_DIM, FUSION_DIM, NUM_HEADS, NUM_CLASSES).to(device)
fusion.load_state_dict(ckpt_f['fusion'])
fusion.eval()
print(f'Fusion loaded. Params: {sum(p.numel() for p in fusion.parameters()):,}')

## Загрузка данных (speaker_text из TSV)

In [ ]:
df_tsv = pd.read_csv(TEST_TSV, sep='\t')
df_tsv = df_tsv[df_tsv['aggregated_emo'].isin(DUSHA_LABEL2ID)]
df_tsv = df_tsv[df_tsv['speaker_text'].notna() & (df_tsv['speaker_text'].str.strip() != '')]
print(f'Test records: {len(df_tsv)}')

MAX_AUDIO_LEN = int(MAX_AUDIO_S * SR_TARGET)

records = []
missing = 0
for _, row in tqdm(df_tsv.iterrows(), total=len(df_tsv), desc='Loading audio'):
    path = AUDIO_TEST / row['audio_path']
    if not path.exists():
        missing += 1; continue
    try:
        wav, sr = sf.read(str(path), dtype='float32')
        if wav.ndim > 1: wav = wav.mean(axis=1)
        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
        records.append({
            'text':    row['speaker_text'],
            'audio':   wav[:MAX_AUDIO_LEN],
            'label':   DUSHA_LABEL2ID[row['aggregated_emo']],
            'emotion': row['aggregated_emo'],
        })
    except Exception: missing += 1

print(f'Loaded: {len(records)}  missing: {missing}')

## Вычисление α

In [ ]:
@torch.no_grad()
def encode_text(texts):
    enc = bert_tokenizer(texts, truncation=True, padding='max_length',
                         max_length=MAX_TEXT_LEN, return_tensors='pt')
    out = bert_backbone(enc['input_ids'].to(device), enc['attention_mask'].to(device))
    return out.last_hidden_state[:, 0, :]

@torch.no_grad()
def encode_audio_single(wav_np):
    inp    = wavlm_processor(wav_np, sampling_rate=SR_TARGET, return_tensors='pt')
    hidden = wavlm_backbone(inp['input_values'].to(device)).last_hidden_state
    return hidden.mean(dim=1)


results = []
for i in tqdm(range(0, len(records), BATCH_SIZE), desc='Computing α'):
    batch    = records[i:i + BATCH_SIZE]
    texts    = [r['text']    for r in batch]
    labels   = [r['label']   for r in batch]
    emotions = [r['emotion'] for r in batch]

    h_text  = encode_text(texts)
    h_audio = torch.cat([encode_audio_single(r['audio']) for r in batch], dim=0)
    logits, alpha_t = fusion.forward_with_alpha(h_text, h_audio)
    probs = F.softmax(logits, dim=-1)

    for j in range(len(batch)):
        at = alpha_t[j].cpu().numpy()
        results.append({
            'text':            texts[j],
            'emotion':         emotions[j],
            'label':           labels[j],
            'pred':            logits[j].argmax().item(),
            'correct':         logits[j].argmax().item() == labels[j],
            'confidence':      float(probs[j].max()),
            'alpha_text_mean': float(at.mean()),
            'alpha_text_vec':  at,
        })

df       = pd.DataFrame([{k: v for k, v in r.items() if k != 'alpha_text_vec'} for r in results])
alpha_mx = np.stack([r['alpha_text_vec'] for r in results])
print(f'Done. α matrix: {alpha_mx.shape}')
print(f'Accuracy={accuracy_score(df.label, df.pred):.4f}  WAcc={balanced_accuracy_score(df.label, df.pred):.4f}')

## Глобальное распределение α

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(alpha_mx.flatten(), bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(0.5, color='red', linestyle='--', label='α=0.5')
axes[0].set_xlabel('α_text(d)'); axes[0].set_title('Распределение α_text (все сэмплы × все размерности)')
axes[0].legend()

axes[1].hist(df['alpha_text_mean'], bins=40, color='darkorange', edgecolor='white', linewidth=0.3)
axes[1].axvline(df['alpha_text_mean'].mean(), color='red', linestyle='--',
                label=f'mean={df["alpha_text_mean"].mean():.3f}')
axes[1].set_xlabel('mean(α_text) per sample'); axes[1].set_title('Среднее α_text на сэмпл')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_global_dusha.png', dpi=150); plt.show()
print(f'mean α_text={alpha_mx.mean():.3f}  mean α_audio={1-alpha_mx.mean():.3f}')

## Примеры с высоким и низким α_text

In [ ]:
df_s = df.sort_values('alpha_text_mean')
print('=== Аудио доминирует (α_text низкое) ===')
for _, row in df_s.head(5).iterrows():
    mark = '✓' if row['correct'] else '✗'
    print(f'  [{mark}] α={row["alpha_text_mean"]:.3f}  true={row["emotion"]:10s}  '
          f'pred={DUSHA_LABELS[int(row["pred"]):int(row["pred"])+1][0]:10s}  "{row["text"][:60]}"')
print()
print('=== Текст доминирует (α_text высокое) ===')
for _, row in df_s.tail(5).iterrows():
    mark = '✓' if row['correct'] else '✗'
    print(f'  [{mark}] α={row["alpha_text_mean"]:.3f}  true={row["emotion"]:10s}  '
          f'pred={DUSHA_LABELS[int(row["pred"]):int(row["pred"])+1][0]:10s}  "{row["text"][:60]}"')

## α по классам эмоций

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = df.groupby('emotion')['alpha_text_mean'].median().sort_values().index.tolist()
sns.boxplot(data=df, x='emotion', y='alpha_text_mean', order=order, palette='coolwarm', ax=ax)
ax.axhline(0.5, color='black', linestyle='--', alpha=0.5, label='α=0.5')
ax.set_title('α_text по классам эмоций (DUSHA)'); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_by_class_dusha.png', dpi=150); plt.show()
print(df.groupby('emotion')['alpha_text_mean'].agg(['median','mean','std']).round(3).to_string())

## Верные vs неверные + уверенность

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for correct, label, color in [(True,'Верные','green'),(False,'Неверные','red')]:
    vals = df[df['correct']==correct]['alpha_text_mean']
    axes[0].hist(vals, bins=30, alpha=0.6, label=f'{label} (n={len(vals)})', color=color)
axes[0].set_xlabel('mean(α_text)'); axes[0].set_title('α_text: верные vs неверные'); axes[0].legend()

axes[1].scatter(df['alpha_text_mean'], df['confidence'],
                c=df['correct'].map({True:'green',False:'red'}), alpha=0.3, s=10)
axes[1].set_xlabel('mean(α_text)'); axes[1].set_ylabel('Confidence')
axes[1].set_title('Уверенность vs α_text')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color='green',label='Верно'),Patch(color='red',label='Неверно')])
plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_correct_dusha.png', dpi=150); plt.show()
print(df.groupby('correct')['alpha_text_mean'].agg(['mean','median','std']).round(3))

## Стабильные «текстовые» и «аудио» размерности

In [ ]:
dim_mean   = alpha_mx.mean(axis=0)
dim_std    = alpha_mx.std(axis=0)
idx_sorted = np.argsort(dim_mean)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(dim_mean[idx_sorted], color='steelblue')
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5)
axes[0].fill_between(range(len(dim_mean)),
                     dim_mean[idx_sorted]-dim_std[idx_sorted],
                     dim_mean[idx_sorted]+dim_std[idx_sorted],
                     alpha=0.2, color='steelblue')
axes[0].set_xlabel('Размерность (отсортировано)'); axes[0].set_ylabel('mean α_text')
axes[0].set_title('Среднее α_text по размерностям (±std)')

top40 = np.concatenate([idx_sorted[:20], idx_sorted[-20:]])
per_class = np.array([
    alpha_mx[df['label'].values == cls, :][:, top40].mean(axis=0)
    for cls in range(NUM_CLASSES)
])
sns.heatmap(per_class, ax=axes[1],
            xticklabels=[f'd{i}' for i in top40], yticklabels=DUSHA_LABELS,
            cmap='RdYlGn', vmin=0, vmax=1, cbar_kws={'label': 'α_text'})
axes[1].set_title('α_text по классам × топ-40 размерностей')
plt.tight_layout()
plt.savefig(OUT_DIR / 'alpha_dimensions_dusha.png', dpi=150); plt.show()

print(f'Доля размерностей α_text>0.5 : {(dim_mean>0.5).mean():.1%}')
print(f'Корреляция (α_text, confidence): {np.corrcoef(df.alpha_text_mean, df.confidence)[0,1]:.3f}')